# 01. 기초 — ADK 그래프의 노드, 엣지, 이벤트

이 실습은 영상의 최소 2노드 그래프를 현재 Google ADK 2.x API로 재구성합니다.

## 학습 목표

1. 함수 노드와 엣지가 어떤 역할을 하는지 설명한다.
2. START에서 두 결정론적 함수로 이어지는 Workflow를 실행한다.
3. Runner가 내보내는 Event를 읽고 모델 호출 예산을 계산한다.

영상은 ADK 2.0을 소개하고 연결 Codelab은 2.3.0을 고정하지만, 이 노트북은
2026-09-11 기준 안정판인 google-adk 2.8.0으로 검증했습니다. Gemini나 API 키는
사용하지 않습니다.


## 실행 준비

Python 3.10 이상이 필요합니다. 아래 import가 실패하면 커널에서 최초 한 번
%pip install google-adk==2.8.0 을 실행한 뒤 커널을 재시작하세요. 버전을 고정하는
이유는 ADK 2.x도 빠르게 변하며 그래프 API의 세부 동작이 달라질 수 있기 때문입니다.


In [ ]:
from importlib.metadata import version

ADK_VERSION = version("google-adk")
assert ADK_VERSION == "2.8.0", (
    f"검증 버전은 2.8.0입니다. 현재 {ADK_VERSION}이므로 "
    "google-adk==2.8.0을 설치해 재현하세요."
)
print("google-adk", ADK_VERSION)


## 1. 두 함수 노드 만들기

첫 노드는 세션 입력의 마라톤 목표를 정규화하고, 둘째 노드는 그 결과로 체크리스트를
만듭니다. 두 작업 모두 규칙으로 완전히 설명할 수 있으므로 LLM을 쓰지 않습니다.
각 함수는 계산 결과를 Event(output=...)로 반환합니다.


In [ ]:
from google.adk import Event, Runner, Workflow
from google.adk.sessions import InMemorySessionService
from google.genai import types


def normalize_goal(node_input):
    # START 바로 다음 함수는 사용자의 Content 메시지를 입력으로 받습니다.
    raw_goal = node_input.parts[0].text
    normalized = " ".join(raw_goal.strip().lower().split())
    return Event(output={"goal": normalized})


def build_checklist(node_input):
    # 순차 엣지에서는 앞 노드의 output이 다음 노드의 입력이 됩니다.
    goal = node_input["goal"]
    items = ["날씨 확인", "코스 고도 확인", "최근 훈련량 확인"]
    return Event(output={"goal": goal, "checklist": items})


## 2. 엣지로 실행 순서 선언하기

START는 외부 입력이 그래프로 들어오는 가상 시작점입니다. 하나의 tuple에 노드를
나열하면 START → normalize_goal → build_checklist 순서가 됩니다.


In [ ]:
workflow = Workflow(
    name="race_preparation",
    edges=[("START", normalize_goal, build_checklist)],
)

edge_table = [
    {
        "from": edge.from_node.name,
        "to": edge.to_node.name,
        "route": edge.route,
    }
    for edge in workflow.graph.edges
]
edge_table


## 3. Runner로 오프라인 실행하기

SessionService는 실행 이력과 상태를 보관합니다. 여기서는 학습용 메모리 저장소를
사용하므로 프로세스를 끝내면 사라집니다. 새 사용자 메시지는 실행을 시작하는
트리거이자 첫 함수 노드의 입력입니다.


In [ ]:
async def run_once(race_goal):
    service = InMemorySessionService()
    session = await service.create_session(
        app_name="graph_learning_lab",
        user_id="learner",
        session_id="foundations",
    )
    runner = Runner(
        node=workflow,
        app_name="graph_learning_lab",
        session_service=service,
    )
    message = types.Content(
        role="user",
        parts=[types.Part(text=race_goal)],
    )

    observed = []
    async for event in runner.run_async(
        user_id="learner",
        session_id=session.id,
        new_message=message,
    ):
        observed.append(
            {
                "author": event.author,
                "output": event.output,
                "error_code": event.error_code,
            }
        )
    return observed


events = await run_once("  첫 풀 마라톤을   안전하게 완주  ")
events


In [ ]:
assert len(events) == 2
assert events[0]["output"]["goal"] == "첫 풀 마라톤을 안전하게 완주"
assert events[-1]["output"]["checklist"] == [
    "날씨 확인",
    "코스 고도 확인",
    "최근 훈련량 확인",
]
assert all(event["error_code"] is None for event in events)
print("검증 통과:", events[-1]["output"])


## 4. 호출 비용을 구조에서 읽기

그래프 노드 수와 LLM 호출 수는 같지 않습니다. 함수, 캐시 조회, 데이터베이스,
JoinNode는 모두 노드가 될 수 있지만 모델을 호출하지 않을 수 있습니다. 아래 표는
실행 전에 예상 호출 수를 계산하는 가장 작은 형태입니다.


In [ ]:
node_inventory = [
    {"name": "normalize_goal", "kind": "function", "llm_calls": 0},
    {"name": "build_checklist", "kind": "function", "llm_calls": 0},
]

current_calls = sum(node["llm_calls"] for node in node_inventory)
with_advice_agent = node_inventory + [
    {"name": "advise", "kind": "llm_agent", "llm_calls": 1}
]
planned_calls = sum(node["llm_calls"] for node in with_advice_agent)

assert current_calls == 0
assert planned_calls == 1
{"현재 오프라인 그래프": current_calls, "조언 Agent 추가 시": planned_calls}


## 핵심 정리와 미니 과제

- 규칙으로 설명 가능한 작업은 함수 노드로 두면 테스트와 비용 예측이 쉽습니다.
- Event는 노드의 출력과 오류, 경로 같은 실행 정보를 Runner에 전달합니다.
- 엣지는 데이터 의존성과 실행 순서를 코드 밖에서 읽을 수 있게 만듭니다.

미니 과제: race_goal의 공백뿐 아니라 대소문자, 빈 문자열을 검증하도록
normalize_goal을 확장하고, 빈 입력일 때 명시적인 오류 Event를 설계해 보세요.

다음: [02_practice.ipynb](02_practice.ipynb)에서 fan-out, JoinNode, router를
실제 ADK 그래프로 연결합니다.
